In [2]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

2.13.0+cu130
True
NVIDIA GB10


In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

In [4]:
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [21]:
ds.column_names

{'test': ['text'], 'train': ['text'], 'validation': ['text']}

In [26]:
print(ds['train'].features)
print(ds['train'].column_names)

{'text': Value(dtype='string', id=None)}
['text']


In [28]:
repr(ds['train'][:15])

'{\'text\': [\'\', \' = Valkyria Chronicles III = \\n\', \'\', \' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \\n\', " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as maki

In [41]:
sum(1 for t in ds['train'][:1000]['text'] if t.strip().startswith("="))

175

In [43]:
def keep_row(example):
    text = example["text"].strip()
    return text != "" and not text.startswith("=")

ds = ds.filter(keep_row)

print(len(ds["train"]))   # expect ~17k, down from 36,718

Filter:   0%|          | 0/4358 [00:00<?, ? examples/s]

Filter:   0%|          | 0/36718 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3760 [00:00<?, ? examples/s]

17556


In [45]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

/home/ankitanand/Documents/pp/Finetuning_HF/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [46]:
tokenizer.pad_token = tokenizer.eos_token

In [48]:
encoded_input = tokenizer("Knowledge distillation is fun")
encoded_input

{'input_ids': [23812, 2965, 1233, 40903, 318, 1257], 'attention_mask': [1, 1, 1, 1, 1, 1]}

In [50]:
tokenizer.tokenize("Knowledge distillation is fun")

['Know', 'ledge', 'Ġdist', 'illation', 'Ġis', 'Ġfun']

In [49]:
tokenizer.decode(encoded_input["input_ids"])

'Knowledge distillation is fun'

In [53]:
encoded_dataset = ds.map(lambda examples: tokenizer(examples["text"]), batched=True, remove_columns="text")

Map:   0%|          | 0/2183 [00:00<?, ? examples/s]

Map:   0%|          | 0/17556 [00:00<?, ? examples/s]

Map:   0%|          | 0/1841 [00:00<?, ? examples/s]

In [56]:
encoded_dataset["train"][0]

{'input_ids': [2311,
  73,
  13090,
  645,
  569,
  18354,
  7496,
  513,
  1058,
  791,
  47398,
  17740,
  357,
  4960,
  1058,
  10545,
  230,
  99,
  161,
  254,
  112,
  5641,
  44444,
  9202,
  25084,
  24440,
  12675,
  11839,
  18,
  837,
  6578,
  764,
  569,
  18354,
  7496,
  286,
  262,
  30193,
  513,
  1267,
  837,
  8811,
  6412,
  284,
  355,
  569,
  18354,
  7496,
  17740,
  6711,
  2354,
  2869,
  837,
  318,
  257,
  16106,
  2597,
  2488,
  12,
  31,
  2712,
  2008,
  983,
  4166,
  416,
  29490,
  290,
  6343,
  13,
  44206,
  329,
  262,
  14047,
  44685,
  764,
  28728,
  287,
  3269,
  2813,
  287,
  2869,
  837,
  340,
  318,
  262,
  2368,
  983,
  287,
  262,
  569,
  18354,
  7496,
  2168,
  764,
  12645,
  278,
  262,
  976,
  21748,
  286,
  16106,
  290,
  1103,
  2488,
  12,
  31,
  640,
  11327,
  355,
  663,
  27677,
  837,
  262,
  1621,
  4539,
  10730,
  284,
  262,
  717,
  983,
  290,
  5679,
  262,
  366,
  17871,
  5321,
  366,
  837,
  257,
  

In [57]:
block_size = 128

In [60]:
def group_texts(batch):
    concatenated = sum(batch["input_ids"], [])
    total_length = (len(concatenated) // block_size) * block_size
    blocks = []
    for i in range(0, total_length, block_size):
        blocks.append(concatenated[i : i + block_size])
    return {"input_ids": blocks, "labels": [b.copy() for b in blocks]}

In [61]:
lm_dataset = encoded_dataset.map(
    group_texts,
    batched=True,
    remove_columns=encoded_dataset["train"].column_names,
)

Map:   0%|          | 0/2183 [00:00<?, ? examples/s]

Map:   0%|          | 0/17556 [00:00<?, ? examples/s]

Map:   0%|          | 0/1841 [00:00<?, ? examples/s]

In [62]:
print(len(lm_dataset["train"]))              # how many blocks? (~16k?)
print(len(lm_dataset["train"][0]["input_ids"]))   # should be exactly 128

18200
128


In [63]:
tokenizer.decode(lm_dataset["train"][0]["input_ids"])

' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3, lit. Valkyria of the Battlefield 3 ), commonly referred to as Valkyria Chronicles III outside Japan, is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable. Released in January 2011 in Japan, it is the third game in the Valkyria series. Employing the same fusion of tactical and real @-@ time gameplay as its predecessors, the story runs parallel to the first game and follows the " Nameless ",'

In [65]:
teacher = AutoModelForCausalLM.from_pretrained("openai-community/gpt2-medium")

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [66]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [67]:
teacher.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1024)
    (wpe): Embedding(1024, 1024)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2SdpaAttention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=1024, out_features=50257, bias=False)
)

In [68]:
for p in teacher.parameters():
    p.requires_grad = False

In [69]:
# rough shape check
out = teacher(torch.tensor([lm_dataset["train"][0]["input_ids"]]).to(device))
out.logits.shape

RuntimeError: Expected all tensors to be on the same device, but got index is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA__index_select)